In [1]:
import sys 
import xarray as xr
import numpy as np
import os
import pandas as pd
import yaml
import math
from datetime import datetime, timedelta
from parcels import FieldSet, Field, VectorField, ParticleSet, JITParticle, ParcelsRandom, Variable,StatusCode,GeographicPolar, Geographic
from glob import glob
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings(
    "ignore",
    message="The specified chunks separate the stored chunks",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message="File .* could not be decoded properly by xarray",
    category=UserWarning,
)

In [2]:
sys.path.append(os.getcwd()+'/Source')

from OP_Kernels import *
from OP_functions import *

In [3]:
ds, name = zarr_tonet('/home/jvalenti/MOAD/results/test_function.zarr')
print(ds.status.isel(obs=-1).to_series().value_counts())   # fate of particles at the end
print(float(ds.ws.max()), float(ds.ws.min()))                # settling velocities
print(ds.N_hb.max().values, ds.N_ph.max().values)            # fouling counts
print(ds.z.min().values, ds.z.max().values)
print(int(ds.lon.isnull().sum()), int(ds.lat.isnull().sum()))  # NaNs = deleted/lost particles

/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'erddapy' loading failed:
cannot import name 'T_PathFileOrDataStore' from 'xarray.backends.common' (/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/xarray/backends/common.py)
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)


status
1.0    273
3.0     70
2.0     21
Name: count, dtype: int64
0.0005118513945490122 5.213025360717438e-05
69992.93 12.971578
0.0 219.77647749176475
0 0


In [3]:
ds, name = zarr_tonet('/home/jvalenti/MOAD/results/test_function.zarr')
print(ds.status.isel(obs=-1).to_series().value_counts())   # fate of particles at the end
print(float(ds.ws.max()), float(ds.ws.min()))                # settling velocities
print(ds.N_hb.max().values, ds.N_ph.max().values)            # fouling counts
print(ds.z.min().values, ds.z.max().values)
print(int(ds.lon.isnull().sum()), int(ds.lat.isnull().sum()))  # NaNs = deleted/lost particles

/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'erddapy' loading failed:
cannot import name 'T_PathFileOrDataStore' from 'xarray.backends.common' (/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/xarray/backends/common.py)
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)


status
1.0    266
3.0     86
2.0     12
Name: count, dtype: int64
0.0005044625140726566 4.800273018190637e-05
48496.26 10.835378
0.0 750.7942652363454
0 0


In [4]:
ds = ds.load()
dmax = ds.z.max('obs')
bad = dmax > 450
print(int(bad.sum()), "trajectories deeper than 450 m")

print(ds.z.isel(obs=0).max().values)           # already deep at release?
print(ds.status.where(bad, drop=True).isel(obs=-1).to_series().value_counts())

i = int(dmax.argmax())
print(ds.z.isel(trajectory=i).values)          # depth vs time for the worst one
print(ds.lat.isel(trajectory=i).values[[0, -1]], ds.lon.isel(trajectory=i).values[[0, -1]])

0 trajectories deeper than 450 m
184.49826915456245
Series([], Name: count, dtype: int64)
[184.08600951 190.55888333 190.03369176 191.27775117 189.38251628
 192.08505358 185.33584247 187.88148929 187.96864479 186.56428698
 188.35180585 191.03020855 188.57526559 184.34404116 182.64055986
 187.96770085 203.93311724 205.65412178 202.84207558 203.41301191
 207.61841939 205.96342373 206.34347668 202.55861803 202.46763857
 201.64221087 199.33248925 201.49411506 209.13773894 202.29846693
 208.19224932 211.97159153 209.76502882 218.05129523 214.67632563
 214.20110205 215.06003658 219.64361281 219.38013612 219.77647749
 180.75634492 180.33306732 179.46070932 178.18326805 174.32855188
 169.44998274 166.99403677 164.62983903]
[47.60059157 47.59295386] [-122.42655131 -122.45896369]


In [5]:
jjii = xr.open_dataset('~/MOAD/grid/grid_from_lat_lon_mask999.nc')

def finder(jjii,lati,loni):
    j = [jjii.jj.sel(lats=lati, lons=loni, method='nearest').item()][0]
    i = [jjii.ii.sel(lats=lati, lons=loni, method='nearest').item()][0]
    return j,i

finder(jjii,47.48715298,-122.36769322)

(49, 230)

In [6]:
dep = ds.z.isel(trajectory=i).values
k = int(np.argmax(dep > 450))          # first bad obs
print(k, ds.status.isel(trajectory=i).values[k-1:k+2])
print(ds.lat.isel(trajectory=i).values[k-1:k+1], ds.lon.isel(trajectory=i).values[k-1:k+1])
print(ds.fact.isel(trajectory=i).values[k-2:k+1])   
print(ds.ws.isel(trajectory=i).values[k-2:k+1])   

0 []
[] []
[]
[]


In [7]:
import numpy as np, xarray as xr
from datetime import datetime, timedelta
from run_Parcels import build_fieldset          # or paste your imports
from OP_functions import *

param = load_config('yaml/config.yaml')
paths = path(param['param']['machine'])
cs = {"time": ("time", 1), "depth": ("depth", 40), "lat": ("latitude", 100), "lon": ("longitude", 100)}
start = datetime(2023, 1, 2)
fs = build_fieldset(start, timedelta(days=1), cs, machine=param['param']['machine'])

# --- the 3 bad trajectories from the output zarr ---
ds = xr.open_zarr('/home/jvalenti/MOAD/results/test_function.zarr').load()
bad = np.where(ds.z.max('obs').values > 450)[0]

def sample(t, z, la, lo):
    out = {}
    for name in ['vert_eddy_diff', 'totaldepth', 'sossheig', 'sigma_theta']:
        try:
            out[name] = float(getattr(fs, name)[t, z, la, lo])
        except Exception as e:
            out[name] = type(e).__name__
    try:
        out['UVW'] = tuple(float(x) for x in fs.UVW[t, z, la, lo])
    except Exception as e:
        out['UVW'] = type(e).__name__
    return out

# 1) at the last good position (obs 7) and the position at obs 8, in time
for i in bad:
    k = int(np.argmax(ds.z.isel(trajectory=i).values > 450))
    for obs in (k-1, k):
        la = float(ds.lat.isel(trajectory=i, obs=obs)); lo = float(ds.lon.isel(trajectory=i, obs=obs))
        t = obs * 1800.0
        print(i, obs, "z=5 ->", sample(t, 5.0, la, lo))

In [9]:
# t = obs * 1800.0
# fs.computeTimeChunk(t, 1)      # load the time slices around t before sampling
# print(i, obs, "z=5 ->", sample(t, 5.0, la, lo))